# LAB 08 — Lakeflow Jobs: Triggers, Dependencies & Orchestration

> **Ticket DATA-240** · From: Head of Data Platform · Priority: High
>
> The Bronze → Silver notebooks of `materials/medallion` still run by hand. The business wants them to run as jobs with dependencies: the Silver customers table refreshes only after Bronze, and the orders processing starts by itself as soon as the customer data is updated (table update trigger).

**Definition of done**
- `<your_name>_customer_pipeline` runs `bronze_customer` → `silver_customer`.
- `<your_name>_orders_pipeline` runs `bronze_orders` → `silver_orders` → `gold_daily_orders` → `gold_summary` and has a table update trigger on `silver.silver_customers`.
- One **Run now** of the customer job refreshes `silver.silver_customers`, and the orders run that the trigger starts refreshes `silver.silver_orders_cleaned` and the Gold tables.

Section 1 is done when **Check your work** in Task 5 of the Workshop Guide app passes. Section 2 is done when the validation cells of Tasks A and B print `Task X OK`. Tasks C, D, E and the Closing Check are stretch tasks. Do them if you finish early, or after the course. Tasks A and B do not need them.

**Goal:** Build a real multi-task Lakeflow Job and change it step by step. You add a `for_each` task and a condition task, set a table update trigger, repair a failed run and compute a success rate from the run history. After each change you check the job with the Databricks SDK.

**Prerequisites:**
- Medallion notebooks (`materials/medallion/`) in your course folder

| Task | What you do | Where |
|------|-------------|-------|
| Section 1 (Steps 1–8) | Create `customer_pipeline` and `orders_pipeline`, add a table update trigger and run the pipeline | UI |
| A | Add a `for_each` task over a list parameter and check it with the SDK | UI + notebook |
| B | Add an If/else condition task and check it | UI + notebook |
| C (stretch) | Move the table update trigger to `bronze_customers` and check it | UI + notebook |
| D (stretch) | Break a task, repair the failed run and check `repair_history` | UI + notebook |
| E (stretch) | Compute the success rate of the job from the run history | notebook |
| Closing check (stretch) | Write a Quartz cron expression and choose the compute type | notebook |

> **SOLUTION NOTEBOOK:** full answers for `lab_08_orchestration`.
>
> **Job dependency:** all Section 2 cells need the Section 1 jobs
> (`<your_name>_customer_pipeline`, `<your_name>_orders_pipeline`) with the UI changes of Tasks A–D:
> the `for_each` and condition tasks added, the trigger moved to Bronze and one run repaired.

## Setup

In [ ]:
%run ../setup/00_setup

In [ ]:
# PROVIDED — run as-is (no changes needed)
# Step 7 puts a table update trigger on silver.silver_customers, and a trigger needs an existing table.
# On a fresh catalog, build it once with the same notebooks that the customer job runs.
MEDALLION = "../../materials/medallion"
silver_customers_table = f"{CATALOG}.{SILVER_SCHEMA}.silver_customers"
if spark.catalog.tableExists(silver_customers_table):
    print(f"[OK] {silver_customers_table} exists")
else:
    dbutils.notebook.run(f"{MEDALLION}/bronze_customers", 900, {"catalog": CATALOG, "source_path": DATASET_PATH})
    dbutils.notebook.run(f"{MEDALLION}/silver_customers", 900, {"catalog": CATALOG})
    print(f"[CREATED] {silver_customers_table} (your customer job refreshes it in Step 8)")

## Section 1 — Workshop: Creating & Running Jobs in the Databricks UI

> **Do Section 1 in the Workshop Guide app** (left window): Lab 08, Tasks 1–5 (notebook Steps 1–8). The app gives you the values to copy for your catalog, hints and **Check your work**. The steps and screenshots below are the same. Use them when the app is not available.

You create two Lakeflow Jobs in the Databricks UI:

| Job | Tasks | Purpose |
|-----|-------|---------|
| **customer_pipeline** | `bronze_customer` → `silver_customer` | Load and clean the customer data |
| **orders_pipeline** | `bronze_orders` → `silver_orders` → `gold_daily_orders` → `gold_summary` | Take the orders through all medallion layers. A customer update starts it |

> **Goal:** Create both jobs and set the task dependencies. Then add a **Table Update trigger**, so that the orders job starts by itself when `silver_customers` is updated, and run the whole pipeline once.

### Step 1: Create a New Job — `customer_pipeline`

A job groups tasks that run together. In the left sidebar, select **Jobs & Pipelines**, then select **Create new** → **Job**.

![Create Job](../../assets/images/training_2026/day3/6f524625f8464c29a2572b0a324333a5.webp)

### Step 2: Set the Job Name

Enter a name for the job, for example `<your_name>_customer_pipeline`. Section 2 finds your jobs by this name.

![Job Name](../../assets/images/training_2026/day3/ae8ab8cafdcf4690b5d9665cbf058798.webp)

> **Note:** The job name does not need to be globally unique. Databricks gives every job a unique **Job ID**.

![Job Details — ID & Creator](../../assets/images/training_2026/day3/460925af2b214f93a2252aed0ea86936.webp)

### Step 3: Configure Job Parameters

Job parameters are passed to every task, so all the notebooks read the same catalog and source path. Add these two **job parameters**:

| Parameter | Value |
|-----------|-------|
| `catalog` | `retailhub_<your_name>` |
| `source_path` | `/Volumes/retailhub_<your_name>/default/datasets` |

![Job Parameters](../../assets/images/training_2026/day3/e24f8fbe50f642e0a28054972140dcb1.webp)

### Step 4: Add the First Task — `bronze_customer`

Each task runs one notebook. Select **Add task** and set these fields:

| Field | Value |
|-------|-------|
| **Task name** | `bronze_customer` |
| **Type** | Notebook |
| **Source** | Workspace |
| **Path** | `<your course folder>/materials/medallion/bronze_customers` (in the Path dialog: Users → your e-mail → your course folder) |
| **Compute** | **Serverless**. It is the default for notebook tasks, so there is no cluster to pick. The training workspace uses serverless. On classic compute you would choose a **Job cluster**, never a shared all-purpose cluster |

![Add Task](../../assets/images/training_2026/day3/8d2245e891b849e5966aae38a0e33c5f.webp)

![Task Configuration](../../assets/images/training_2026/day3/c89c69190d40463faae2b5186c78c271.webp)

Select **Create task** when you are done.

### Step 5: Add the Second Task — `silver_customer`

**Depends on** sets the order of the tasks. Add the task `silver_customer` in the same way:
- Set **Depends on** to `bronze_customer`, so that the task runs only after the Bronze task has succeeded.
- Set **Path** to `<your course folder>/materials/medallion/silver_customers`.

The finished job looks like this:

![Customer Pipeline — 2 Tasks](../../assets/images/training_2026/day3/8bdeb6d46c284e3c85bf1c04f4b27657.webp)

### Step 6: Create the Second Job — `orders_pipeline`

The orders job takes the orders through all medallion layers. It has four notebook tasks in a chain, and each task depends on the one before it.

Create a **new job** in the same way. Name it `<your_name>_orders_pipeline`, add the same job parameters as in Step 3 (`catalog`, `source_path`) and add these four Notebook tasks on Serverless compute:

| Task key | Notebook path | Depends on |
|----------|---------------|------------|
| `bronze_orders` | `<your course folder>/materials/medallion/bronze_orders` | — |
| `silver_orders` | `<your course folder>/materials/medallion/silver_orders_cleaned` | `bronze_orders` |
| `gold_daily_orders` | `<your course folder>/materials/medallion/gold_daily_orders` | `silver_orders` |
| `gold_summary` | `<your course folder>/materials/medallion/gold_customer_orders_summary` | `gold_daily_orders` |

> Use exactly these task keys. Section 2 refers to `gold_summary`.

The finished job looks like this:

![Orders Pipeline — 4 Tasks](../../assets/images/training_2026/day3/119b8804bea74938aff23d816140bf4b.webp)

### Step 7: Configure a Table Update Trigger

A table update trigger starts a job when a Unity Catalog table changes. You use it when a job must run right after another job has written its data, without a schedule.

On the `orders_pipeline` job page, open **Schedules & Triggers** and add a trigger of type **Table update** on `retailhub_<your_name>.silver.silver_customers`.

The orders job now **starts by itself** whenever the customer job updates the `silver_customers` table.

![Table Update Trigger Configuration](../../assets/images/training_2026/day3/d4dd6298fc9b41e9bc784878ccd8dc3a.webp)

### Step 8: Run the Pipeline

You start only the customer job. The trigger starts the orders job. Select **Run now** on the `customer_pipeline` job.

![Run Job](../../assets/images/training_2026/day3/73685979cbb243b3af4e01fdcadb1a32.webp)

> **Expected result:** The customer job finishes successfully and updates `silver_customers`. This starts the orders job by itself. Both jobs show a successful run in the **Runs** tab.

## Section 2 — Engineering: Evolve the Job, Verify with the SDK

Each task has two parts. First you change the job in the UI (or in JSON). Then you check the change with code. The validation cells use the [Databricks SDK for Python](https://docs.databricks.com/dev-tools/sdk-python.html) (`databricks.sdk`), which authenticates automatically inside a notebook.

> **Requires Section 1:** the jobs `<your_name>_customer_pipeline` and `<your_name>_orders_pipeline` must exist.

| Task | Topic | Verified by |
|------|-------|-------------|
| A | `for_each` task over a list parameter | SDK: task type in the job settings |
| B | Conditional (If/else) task | SDK: `condition_task` is set |
| C (stretch) | Table update trigger on a Bronze table | Jobs API JSON: `trigger.table_update` |
| D (stretch) | Break a task and use Repair run | SDK: `repair_history` of the run |
| E (stretch) | Run history and success rate | `system.lakeflow` (fallback: Jobs API) |

In [ ]:
# PROVIDED — run as-is (no changes needed)
# SDK client + job lookup helpers
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()  # auto-auth from the notebook context

user_name    = CATALOG.replace(f"{CATALOG_PREFIX}_", "")
CUSTOMER_JOB = f"{user_name}_customer_pipeline"
ORDERS_JOB   = f"{user_name}_orders_pipeline"

def get_job(name):
    """Return the full job object for an exact job name (Section 1 naming)."""
    matches = list(w.jobs.list(name=name))
    if not matches:
        raise ValueError(
            f"No job named '{name}' found. Finish Section 1 first, "
            f"or adjust CUSTOMER_JOB / ORDERS_JOB to your actual job names."
        )
    return w.jobs.get(matches[0].job_id)

# Value of the job parameter tables_to_check (Task A), with your catalog: copy it from the output
import json
TABLES_TO_CHECK = json.dumps([f"{CATALOG}.bronze.bronze_customers", f"{CATALOG}.silver.silver_customers", f"{CATALOG}.gold.gold_daily_orders"])

print(f"SDK connected as : {w.current_user.me().user_name}")
print(f"Expected jobs    : {CUSTOMER_JOB} | {ORDERS_JOB}")
print(f"tables_to_check  : {TABLES_TO_CHECK}")

### Task A — Add a `for_each` Task Over a List Parameter

A `for_each` task runs a nested task once for each element of a list. This is how you write a loop in Lakeflow Jobs, for example to run the same check on several tables. The exam guide of May 2026 lists it as an objective.

**UI steps on `<your_name>_orders_pipeline`:**

Paste the values below into the fields; do not type them. The value fields close brackets by themselves, so a typed `{{...}}` ends with extra brackets.

1. Under **Job parameters**, add the parameter `tables_to_check`. Its value is printed by the first cell of this section, in the line `tables_to_check`, already with your catalog: copy it from there. The names are fully qualified, because the validation notebook reads each table with `spark.table(...)`. The value looks like this:

   ```text
   ["retailhub_<your_name>.bronze.bronze_customers", "retailhub_<your_name>.silver.silver_customers", "retailhub_<your_name>.gold.gold_daily_orders"]
   ```
2. Select **Add task** → **For each**. Set **Task name** to `check_each_table` and **Depends on** to `gold_summary` (or your last task).
3. Set **Inputs** to `{{job.parameters.tables_to_check}}`, the list to loop over, and **Concurrency** to 1.
4. Select **Add a task to loop over** and set up the nested task:
   - 4a. Set **Task name** to `check_one_table` (the UI suggests `check_each_table_iteration`) and **Type** to **Notebook**.
   - 4b. Set **Path** to `<your course folder>/materials/orchestration/task_01_validate`.
   - 4c. Add the task parameter `source_table` = `{{input}}`. In each iteration, `{{input}}` holds one element of the list.
   - 4d. Add the task parameter `min_rows` = `1`.

> **Why `min_rows` = `1`?** The notebook's default is 100. That would fail on `gold_daily_orders`, which has about 32 rows, one per day.

**JSON equivalent** (Job → ⋮ → **View as code** → **JSON**). The UI generates this JSON:
```json
{
  "task_key": "check_each_table",
  "depends_on": [{"task_key": "gold_summary"}],
  "for_each_task": {
    "inputs": "{{job.parameters.tables_to_check}}",
    "concurrency": 1,
    "task": {
      "task_key": "check_one_table",
      "notebook_task": {
        "notebook_path": "/Workspace/.../task_01_validate",
        "base_parameters": {"source_table": "{{input}}", "min_rows": "1"}
      }
    }
  }
}
```

Then replace the two `...` in the code cell so that `orders_job` holds your orders job and `foreach_tasks` lists its `for_each` tasks. The validation cell prints `Task A OK`.

In [ ]:
orders_job = get_job(ORDERS_JOB)
foreach_tasks = [t for t in orders_job.settings.tasks if t.for_each_task is not None]

for t in foreach_tasks:
    print(f"for_each task '{t.task_key}' iterating over: {t.for_each_task.inputs}")

#### Hint — Task A

The SDK follows the JSON. Each element of `job.settings.tasks` is a `Task` object. Its type field (`notebook_task`, `pipeline_task`, `for_each_task`, `condition_task`, ...) is `None` unless the task has that type.

```python
task.for_each_task                 # None unless this task is a for_each task
task.for_each_task.inputs          # the list expression / literal
task.for_each_task.task.task_key   # the nested task
```

In [ ]:
# -- Validation --
assert "orders_job" in globals() and "foreach_tasks" in globals(), "Run the Task A cell above first"
assert orders_job is not ... and ... not in foreach_tasks, "Task A: replace each ... in the cell above with your code, then run it again"
all_keys = [t.task_key for t in orders_job.settings.tasks]
assert len(foreach_tasks) >= 1, \
    f"No for_each task found in '{ORDERS_JOB}'. Tasks present: {all_keys}"
fe = foreach_tasks[0].for_each_task
assert fe.inputs, "for_each task must define 'inputs' (the list to iterate)"
assert fe.task is not None, "for_each task must contain a nested task to run per element"
print(f"Task A OK: '{foreach_tasks[0].task_key}' loops a nested "
      f"'{fe.task.task_key}' task over {fe.inputs}")

### Task B — Add a Conditional (If/else) Task

An **If/else condition** task splits the DAG into two branches. Each downstream task says whether it runs on the `true` or on the `false` outcome. Branching is the other control flow objective of the exam.

**UI steps on `<your_name>_orders_pipeline`:**
1. Under **Job parameters**, add `min_rows_ok` = `true`.
2. Select **Add task** → **If/else condition**. Set **Task name** to `check_row_count` and **Depends on** to your `for_each` task (or `gold_summary`).
3. Set the **Condition**: left value `{{job.parameters.min_rows_ok}}`, operator `==`, right value `true`. In production the left side would be a task value, such as `{{tasks.gold_summary.values.row_count}}`.
4. Add a downstream Notebook task `send_report` with **Path** `<your course folder>/materials/orchestration/task_03_report` and **Depends on** `check_row_count` **(true)**. It runs only on the true branch.
5. Select **Run now** on the job once. In the run, `check_each_table` shows 3 iterations (one per table) and `send_report` runs on the true branch.

**JSON equivalent:**
```json
{
  "task_key": "check_row_count",
  "depends_on": [{"task_key": "check_each_table"}],
  "condition_task": {
    "op": "EQUAL_TO",
    "left": "{{job.parameters.min_rows_ok}}",
    "right": "true"
  }
}
```

Then replace the `...` so that `condition_tasks` lists the If/else tasks of the orders job. The validation cell prints `Task B OK`.

In [ ]:
orders_job = get_job(ORDERS_JOB)
condition_tasks = [t for t in orders_job.settings.tasks if t.condition_task is not None]

for t in condition_tasks:
    ct = t.condition_task
    print(f"condition task '{t.task_key}': {ct.left} {ct.op.value if ct.op else '?'} {ct.right}")

In [ ]:
# -- Validation --
assert "condition_tasks" in globals(), "Run the Task B cell above first"
assert ... not in condition_tasks, "Task B: replace the ... in the cell above with your code, then run it again"
assert len(condition_tasks) >= 1, \
    f"No If/else condition task found in '{ORDERS_JOB}'. " \
    f"Tasks: {[t.task_key for t in orders_job.settings.tasks]}"
ct = condition_tasks[0].condition_task
assert ct.left and ct.right and ct.op, "Condition needs left value, operator, and right value"
dependents = [
    t.task_key for t in orders_job.settings.tasks
    if t.depends_on and any(d.task_key == condition_tasks[0].task_key for d in t.depends_on)
]
print(f"Task B OK: '{condition_tasks[0].task_key}' branches the DAG "
      f"(downstream: {dependents or 'none yet — add a true-branch task!'})")

### Task C — Table-Update Trigger on a Bronze Table (stretch)

> **Stretch:** optional in class. Do it if you finish early, or after the course. Start it only after **Check your work** of Task 5 in the app has passed, because it moves the trigger that Section 1 checks.

Section 1 put the table update trigger on `silver_customers`. Now move it one layer earlier, so that the orders job starts as soon as **`bronze_customers`** is updated.

**UI steps on `<your_name>_orders_pipeline`:**
1. On the job page, open **Schedules & Triggers** and edit the trigger.
2. Keep the type **Table update** and set **Tables** to `retailhub_<your_name>.bronze.bronze_customers`.
3. Select **Save**. Optionally, under **Advanced**, set **Minimum time between triggers in seconds**, so that many quick updates do not start many runs.

> **For the exam:** know all trigger types and when each one fits: Scheduled (cron), **File arrival**, **Table update**, **Model update** and Continuous. Without a trigger, the job runs only when you start it.

Then replace the two `...` so that `trigger` holds the trigger settings of the job and `table_names` the tables it watches. The validation cell prints `Task C OK`.

In [ ]:
orders_job = get_job(ORDERS_JOB)
# PROVIDED — run as-is: raw REST (the serverless SDK may not map table_update)
raw_settings = w.api_client.do("GET", "/api/2.2/jobs/get", query={"job_id": orders_job.job_id})["settings"]
trigger = raw_settings.get("trigger")              # dict, or None when no trigger is configured
table_names = (trigger or {}).get("table_update", {}).get("table_names", [])

print(f"Trigger watches: {table_names}")

In [ ]:
# -- Validation --
table_trigger = (trigger or {}).get("table_update")   # trigger is the raw JSON dict from the cell above
assert trigger is not None, f"'{ORDERS_JOB}' has no trigger configured"
assert table_trigger is not None, "Expected a TABLE UPDATE trigger (not cron/file-arrival)"
assert len(table_names) >= 1, "Table-update trigger must watch at least one table"
assert any(t.lower().startswith(CATALOG.lower()) for t in table_names), \
    f"Trigger should watch a table in YOUR catalog {CATALOG}: {table_names}"
assert any(".bronze." in t.lower() and "bronze_customers" in t.lower() for t in table_names), \
    f"Point the trigger at {CATALOG}.bronze.bronze_customers — currently: {table_names}"
print(f"Task C OK: orders pipeline fires on updates to {table_names}")

### Task D — Break & Repair: Fix a Failed Run Without Re-Running Everything (stretch)

> **Stretch:** optional in class. Do it if you finish early, or after the course. Only Task E (stretch) uses its runs.

A **Repair run** runs again only the failed task and the tasks downstream of it. Tasks that succeeded are skipped. You use it when one task of a long job fails, so that you do not run the whole job again.

**Steps on `<your_name>_orders_pipeline`:**
1. Break the job. The trainer may have done this for you. Edit the `gold_summary` task and change its **notebook path** to a name that does not exist, for example add `_BROKEN` at the end (`.../gold_customer_orders_summary_BROKEN`).
2. Select **Run now**. `bronze_orders`, `silver_orders` and `gold_daily_orders` succeed, and `gold_summary` **fails**. Downstream tasks show *Upstream failed*.
3. Fix the task: restore the original notebook path.
4. On the page of the failed run, select **Repair run**, select the failed task and confirm. Only `gold_summary` and the tasks downstream of it run again.
5. The run now finishes green.

> **Why not break it with a parameter?** A task parameter such as `catalog = retailhub_broken` does not work. **Job parameters override task parameters with the same key**, so the task still gets the job-level `catalog` and succeeds.

Then complete the loop in the code cell so that `repaired_runs` lists the runs of the orders job that were repaired. The validation cell prints `Task D OK`.

In [ ]:
repaired_runs = []
for r in w.jobs.list_runs(job_id=orders_job.job_id, limit=25):
    full = w.jobs.get_run(r.run_id, include_history=True)
    if full.repair_history and len(full.repair_history) > 1:
        repaired_runs.append(full)

for run in repaired_runs:
    print(f"run_id={run.run_id}: {len(run.repair_history)} repair-history entries "
          f"(1 original + {len(run.repair_history) - 1} repair(s)), "
          f"final state: {run.state.result_state.value if run.state.result_state else None}")

#### Hint — Task D

`w.jobs.list_runs(job_id=...)` lists the runs, newest first. `w.jobs.get_run(run_id)` returns the full run with **`repair_history`**, but **only when you call it with `include_history=True`**. Without it, the list comes back empty. The list has one `ORIGINAL` entry plus one entry per repair, so a repaired run has `len(repair_history) > 1`.

```python
w.jobs.get_run(run_id, include_history=True).repair_history   # ORIGINAL entry + one entry per repair
```

In [ ]:
# -- Validation --
assert len(repaired_runs) >= 1, (
    "No repaired run found. Break gold_summary's notebook path, Run now, restore the path, "
    "then use 'Repair run' on the failed run — and re-run this cell."
)
rr = repaired_runs[0]
repair_entries = [h for h in rr.repair_history if h.type and h.type.value == "REPAIR"]
assert len(repair_entries) >= 1, "repair_history should contain at least one REPAIR entry"
print(f"Task D OK: run {rr.run_id} was repaired {len(repair_entries)} time(s) — "
      "succeeded tasks were NOT re-executed")

### Task E — Run History & Success Rate: System Tables (with API Fallback) (stretch)

> **Stretch:** optional in class. Do it if you finish early, or after the course. It needs at least 2 runs of the orders job, for example the two runs of Task D.

Compute the success rate of the job from **`system.lakeflow.job_run_timeline`**. This system table holds the run history, you can query it with SQL, and monitoring dashboards use it. An account admin must enable system tables, and the data can be up to about an hour late. That is why the query runs in `try/except`, with the Jobs API as a fallback.

| Source | Access | Freshness |
|--------|--------|-----------|
| `system.lakeflow.job_run_timeline` | SQL, needs SELECT grant on the schema | Up to about 1 hour of lag |
| `w.jobs.list_runs()` (Jobs API) | Always available to the job owner | Real-time |

Count runs, not rows. `job_run_timeline` can hold several rows for one `run_id`. A run longer than 1 hour is split into hourly rows, and the run that you broke and repaired in Task D has a row for the failed attempt **and** a row for the repair. Group by `run_id` and keep the **final** state with `MAX_BY(result_state, period_end_time)`. The repaired run then counts **once, with its final state** (SUCCEEDED after a successful repair). The Jobs API also returns one entry per run with its final state. System tables cover the whole account, so also filter on `workspace_id`. The cell already has the line `ws_id = w.get_workspace_id()`.

If you see `[INFO] Falling back…`, your system table query failed or returned no rows. Check the SQL and the error printed above it, or wait up to an hour for the system table data. The Jobs API fallback gives the result now.

In [ ]:
result_states, source = [], None
ws_id = w.get_workspace_id()   # system tables are account-wide -> filter on this workspace

try:
    # ONE row per run: hourly rows and repair attempts collapse into the run's FINAL state
    rows = spark.sql(f"""
        WITH runs AS (
            SELECT run_id,
                   MAX(period_end_time)                  AS end_time,
                   MAX_BY(result_state, period_end_time) AS result_state   -- final state
            FROM system.lakeflow.job_run_timeline
            WHERE workspace_id = '{ws_id}'
              AND job_id = '{orders_job.job_id}'
            GROUP BY run_id
        )
        SELECT result_state
        FROM runs
        WHERE result_state IS NOT NULL
        ORDER BY end_time DESC
        LIMIT 50
    """).collect()
    result_states = [r["result_state"] for r in rows]
    source = "system.lakeflow.job_run_timeline"
    if not result_states:
        raise ValueError("system table returned no rows yet (ingestion lag)")
except Exception as e:
    print(f"[INFO] System-table path unavailable ({type(e).__name__}: {e})")
    print("[INFO] Falling back to the Jobs API — one final state per run, real-time.")
    result_states = [
        r.state.result_state.value
        for r in w.jobs.list_runs(job_id=orders_job.job_id, limit=50)
        if r.state and r.state.result_state
    ]
    source = "Jobs API (w.jobs.list_runs)"

total     = len(result_states)
succeeded = sum(1 for s in result_states if s == "SUCCESS" or s == "SUCCEEDED")
success_rate = succeeded / total * 100 if total else 0.0
print(f"Source: {source}")
print(f"Runs analyzed: {total} | Succeeded: {succeeded} | Success rate: {success_rate:.1f}%")

In [ ]:
# -- Validation --
assert total >= 2, \
    f"Expected at least 2 finished runs (Section 1 run + Task D break/repair), got {total}"
assert succeeded >= 1, "Expected at least one successful run"
assert 0 <= success_rate <= 100
assert source is not None
repair_note = " — the repaired Task D run counts once, with its final state" if globals().get("repaired_runs") else ""
print(f"Task E OK: success rate {success_rate:.1f}% over {total} runs (source: {source}){repair_note}")

## Closing Check — Two Exam Classics (stretch)

> **Stretch:** optional in class. Do it if you finish early, or after the course. The same two questions are in the Day-3 quiz.

Two short fill-ins from the concept quiz: cron syntax and the choice of compute.

> **Note:** Lakeflow Jobs schedules use **Quartz cron** syntax with 6 or 7 fields: `seconds minutes hours day-of-month month day-of-week [year]`. For example, `0 0 6 * * ?` runs daily at 06:00. Exactly one of day-of-month and day-of-week must be `?`. The Jobs scheduler does **not** accept the classic 5-field Unix cron (`0 6 * * *`).

In [ ]:
cron_weekdays_8am = "0 0 8 ? * MON-FRI"   # Quartz: sec min hour dom month dow

nightly_etl     = "job_cluster"
interactive_dev = "all_purpose"


In [ ]:
# -- Validation --
_f = cron_weekdays_8am.split()
assert len(_f) in (6, 7), f"Quartz cron has 6-7 fields (seconds first), got {len(_f)}: '{cron_weekdays_8am}'"
assert _f[0] == "0" and _f[1] == "0" and _f[2] == "8", "seconds=0, minutes=0, hours=8"
assert _f[3] == "?" and _f[4] == "*", "day-of-month must be '?' when day-of-week is set; month '*'"
assert _f[5].upper() in ("MON-FRI", "2-6"), "day-of-week: MON-FRI (Quartz numbering 2-6, 1 = SUN)"
assert nightly_etl == "job_cluster", "Scheduled ETL -> job cluster (cheaper, auto-terminates)"
assert interactive_dev == "all_purpose", "Interactive work -> all-purpose (stays running, shared)"
print("Closing check OK: Quartz cron + compute selection correct")


## Reference — job YAML

> The YAML below defines the two Section 1 jobs as a **Declarative Automation Bundle** (formerly Databricks Asset Bundles). It is the code version of what you set up in the UI in Section 1, before the Section 2 changes.

### `orders_pipeline`

```yaml
resources:
  jobs:
    demo_orders_pipeline:
      name: demo_orders_pipeline
      trigger:
        pause_status: UNPAUSED
        table_update:
          table_names:
            - retailhub_trainer.silver.silver_customers
      tasks:
        - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/.../bronze_orders
            source: WORKSPACE
        - task_key: silver_orders
          depends_on:
            - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/.../silver_orders_cleaned
            source: WORKSPACE
        - task_key: gold_daily_orders
          depends_on:
            - task_key: silver_orders
          notebook_task:
            notebook_path: /Workspace/.../gold_daily_orders
            source: WORKSPACE
        - task_key: gold_summary
          depends_on:
            - task_key: gold_daily_orders
          notebook_task:
            notebook_path: /Workspace/.../gold_customer_orders_summary
            source: WORKSPACE
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_trainer
        - name: source_path
          default: /Volumes/retailhub_trainer/default/datasets
```

### `customer_pipeline`

```yaml
resources:
  jobs:
    demo_customer_pipeline:
      name: demo_customer_pipeline
      tasks:
        - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/.../bronze_customers
            source: WORKSPACE
        - task_key: silver_customer
          depends_on:
            - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/.../silver_customers
            source: WORKSPACE
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_trainer
        - name: source_path
          default: /Volumes/retailhub_trainer/default/datasets
```

> **Key differences:** The customer pipeline has **no trigger**: you start it by hand or on demand. The orders pipeline has a **table_update trigger** on `silver_customers`.

## Summary

### Section 1 — Workshop
You created two Lakeflow Jobs in the Databricks UI, set the task dependencies, added a **Table Update trigger** and ran the whole pipeline once.

### Section 2 — Engineering
You changed the job step by step and checked every change with code:

| Task | Skill | Key Takeaway |
|------|-------|-------------|
| A | `for_each` task | Loops a nested task over a list, with `{{input}}` for each element |
| B | If/else condition | `condition_task` splits the DAG, and each downstream task picks the true or false outcome |
| C (stretch) | Table-update trigger | A job can start when a Delta table changes, without polling or cron |
| D (stretch) | Repair run | Runs again only the failed task and the tasks downstream of it. `repair_history` records the repair |
| E (stretch) | Run monitoring | `system.lakeflow.job_run_timeline` for monitoring in SQL. Count runs (final state per `run_id`), not rows. The Jobs API is the real-time fallback |

> **Exam Tip:** Know **when** to use each trigger type. Know that `for_each` and condition tasks are task types of their own, and that a repair run runs again only the failed task and the tasks downstream of it. For scheduled workloads, use serverless or job compute, not an all-purpose cluster.

← <a href="$../day3/demo/08_job_orchestration">08 — Job Orchestration</a> | **<a href="$../..">Course folder (README)</a>** | <a href="$../day3/demo/09_cicd_and_bundles">09 — CI/CD & Bundles →</a>